<a href="https://colab.research.google.com/github/AbuWabu3697/VectorForge-CPU-GPU-Vector-Search-NLP-Benchmarking-Engine/blob/main/notebooks/vectorforge_gpu_profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VectorForge Part 3 — Google Colab GPU profiling

This is the GPU execution environment for developers without a local NVIDIA GPU. It preserves Colab's CUDA PyTorch, clones VectorForge, compiles and tests the custom CUDA kernels, profiles representative workloads, and packages the measured artifacts.

**Before running:** choose **Runtime → Change runtime type → T4 GPU**, then choose **Runtime → Run all**. Push your latest repository changes to GitHub first.

## 1. Require an NVIDIA GPU runtime

In [1]:
import shutil, subprocess, sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('PyTorch CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('nvcc:', shutil.which('nvcc'))
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > T4 GPU, restart, and run again.')
if not shutil.which('nvcc'):
    raise RuntimeError('This runtime has no CUDA compiler, so the custom-kernel tests cannot run.')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['nvcc', '--version'], check=True)

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
PyTorch CUDA runtime: 12.8
CUDA available: True
nvcc: /usr/local/cuda/bin/nvcc
GPU: Tesla T4


CompletedProcess(args=['nvcc', '--version'], returncode=0)

## 2. Clone VectorForge

Opening a GitHub notebook in Colab does not clone its repository into the VM, so this creates a fresh working copy.

In [2]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/AbuWabu3697/VectorForge-CPU-GPU-Vector-Search-NLP-Benchmarking-Engine.git'
PROJECT_DIR = Path('/content/VectorForge-CPU-GPU-Vector-Search-NLP-Benchmarking-Engine')
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print('Reusing existing checkout:', PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Working directory:', Path.cwd())

Commit: 30801ee
Working directory: /content/VectorForge-CPU-GPU-Vector-Search-NLP-Benchmarking-Engine


## 3. Install dependencies without replacing CUDA PyTorch

The installer removes the torch requirement because Colab already provides a compatible CUDA build. Ninja is installed for the custom extension compiler.

In [3]:
requirements = [
    line.strip()
    for line in Path('requirements.txt').read_text().splitlines()
    if line.strip() and not line.strip().startswith('#') and not line.lower().startswith('torch')
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *requirements, 'ninja'], check=True)
import torch
assert torch.cuda.is_available(), 'CUDA disappeared after dependency installation.'
print('CUDA PyTorch preserved:', torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

CUDA PyTorch preserved: 2.11.0+cu128 12.8 Tesla T4


## 4. Run every test, including CUDA

The expected result is 29 passed and zero skipped. The first run compiles both educational CUDA variants.

In [4]:
test_run = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', '-rs'],
    text=True, capture_output=True,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode != 0:
    raise RuntimeError('The test suite failed; inspect the output above.')
if 'skipped' in test_run.stdout.lower():
    raise RuntimeError('At least one test was skipped; this is not complete CUDA validation.')
print('Complete CPU + CUDA validation passed.')

.............................                                            [100%]
29 passed in 86.92s (0:01:26)

Complete CPU + CUDA validation passed.


## 5. Record hardware and demonstrate correct CUDA timing

In [5]:
import json
from src.profiling.common.device_info import collect_device_info
from src.profiling.common.timers import CudaEventTimer, WallClockTimer

summary_dir = Path('results/profiling/summaries')
summary_dir.mkdir(parents=True, exist_ok=True)
hardware = collect_device_info(torch.device('cuda'))
print(json.dumps(hardware, indent=2))
(summary_dir / 'colab-hardware.json').write_text(json.dumps(hardware, indent=2))

x = torch.randn(2048, 2048, device='cuda')
for _ in range(3):
    _ = x @ x
torch.cuda.synchronize()
with WallClockTimer(torch.device('cuda')) as wall:
    _ = x @ x
with CudaEventTimer(torch.device('cuda')) as event:
    _ = x @ x
timing = {'end_to_end_wall_ms': wall.elapsed_ms, 'device_stream_event_ms': event.elapsed_ms}
print(timing)
(summary_dir / 'colab-timing-demo.json').write_text(json.dumps(timing, indent=2))

{
  "device_type": "cuda",
  "device_name": "Tesla T4",
  "gpu_count": 1,
  "torch_version": "2.11.0+cu128",
  "cuda_version": "12.8",
  "compute_capability": "7.5",
  "total_vram_mb": 14912.6875
}
{'end_to_end_wall_ms': 6.28568299998733, 'device_stream_event_ms': 6.093664169311523}


91

## 6. Generate OCR data and profile small/large FP32/FP16 runs

In [6]:
subprocess.run([sys.executable, '-m', 'src.ocr.data.synthetic_generator', '--config', 'config/ocr.yaml'], check=True)
subprocess.run([
    sys.executable, '-m', 'src.cli.profile_ocr',
    '--device', 'cuda',
    '--batch-size', '8', '--batch-size', '64',
    '--precision', 'fp32', '--precision', 'fp16',
    '--epochs', '1',
], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'src.cli.profile_ocr', '--device', 'cuda', '--batch-size', '8', '--batch-size', '64', '--precision', 'fp32', '--precision', 'fp16', '--epochs', '1'], returncode=0)

## 7. Benchmark custom CUDA search

Documents remain resident on GPU. Query H2D, custom scoring, CUDA top-k, and D2H are recorded separately for query batches 1 and 32.

In [7]:
subprocess.run([
    sys.executable, '-m', 'src.profiling.cuda.brute_force_search.benchmark',
    '--dataset-size', '10000',
    '--batch-size', '1', '--batch-size', '32',
    '--output', 'results/profiling/summaries/custom-cuda-colab.csv',
], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'src.profiling.cuda.brute_force_search.benchmark', '--dataset-size', '10000', '--batch-size', '1', '--batch-size', '32', '--output', 'results/profiling/summaries/custom-cuda-colab.csv'], returncode=0)

## 8. Inspect summaries, observations, and real-result plots

In [9]:
import importlib
import src.profiling.analysis.profile_plots as profile_plots

importlib.reload(profile_plots)

plot_path = profile_plots.plot_search_latency_breakdown(
    cuda_frame.to_dict(orient="records"),
    "results/profiling/plots/custom-cuda-breakdown.png",
)

display(Image(filename=str(plot_path)))

AttributeError: 'str' object has no attribute 'get'

## 9. Record tool availability and download evidence

Colab may omit Nsight Systems/Compute. Missing tools are recorded rather than replaced with fabricated reports.

In [ ]:
tools = {'nsys': shutil.which('nsys'), 'ncu': shutil.which('ncu')}
print(tools)
(summary_dir / 'colab-profiler-tools.json').write_text(json.dumps(tools, indent=2))
if not tools['nsys'] or not tools['ncu']:
    print('Using PyTorch traces and CUDA Event measurements; full Nsight collection is unavailable here.')

archive = shutil.make_archive('/content/vectorforge_profiling_results', 'zip', 'results/profiling')
print('Created:', archive)
from google.colab import files
files.download(archive)